# 06 · Caja y capital circulante

Traduce EBITDA en caja operativa y flujo libre, haciendo visible el efecto del circulante y el capex recurrente.

> **Fuente:** datos sintéticos de Levante Ferries. Proyecto demostrativo; no contiene información real de ninguna naviera.

## Contexto y método

El notebook forma parte de una cadena reproducible. Las fórmulas y supuestos se muestran junto a los resultados para que cada conclusión pueda revisarse.

In [1]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
FIGURES = ROOT / 'outputs' / 'figures'
for folder in [PROCESSED, TABLES, FIGURES]: folder.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette=['#1f6feb','#2dd4bf','#f59e0b','#ef4444','#94a3b8'])
plt.rcParams.update({'figure.figsize': (11, 5.5), 'axes.titlesize': 14, 'axes.labelsize': 10})
pd.options.display.float_format = '{:,.2f}'.format


## Conversión de caja

In [2]:
c=pd.read_csv(RAW/'fact_cashflow_monthly.csv',parse_dates=['month'])
c['cash_conversion']=c.operating_cash_flow/c.ebitda.replace(0,np.nan)
annual=c.assign(year=c.month.dt.year).groupby('year',as_index=False).agg(ebitda=('ebitda','sum'),operating_cash_flow=('operating_cash_flow','sum'),free_cash_flow=('free_cash_flow','sum'),maintenance_capex=('maintenance_capex','sum'),avg_nwc=('net_working_capital','mean'))
annual['cash_conversion']=annual.operating_cash_flow/annual.ebitda
annual.to_csv(TABLES/'06_cashflow_annual.csv',index=False)
display(annual)

   year        ebitda  ...    avg_nwc  cash_conversion
0  2024  8,801,538.82  ... 260,810.47             0.47
1  2025 10,932,407.56  ... 275,509.24             0.56
2  2026 19,850,741.30  ... 331,080.11             0.71

[3 rows x 7 columns]


## Evolución de caja

In [3]:
plt.figure(figsize=(12,5)); plt.plot(c.month,c.operating_cash_flow/1e6,label='Caja operativa',color='#1f6feb'); plt.plot(c.month,c.free_cash_flow/1e6,label='Flujo libre',color='#2dd4bf'); plt.axhline(0,color='#ef4444',lw=.8); plt.ylabel('M€'); plt.title('Conversión mensual de EBITDA en caja'); plt.legend()
plt.tight_layout(); plt.savefig(FIGURES/'06_cashflow_trend.png',dpi=180,bbox_inches='tight'); plt.show()

## Meses de tensión

In [4]:
stress=c.nsmallest(6,'free_cash_flow')[['month','ebitda','delta_nwc','maintenance_capex','free_cash_flow']]
stress.to_csv(TABLES/'06_cash_stress_months.csv',index=False)
display(stress)

        month        ebitda  delta_nwc  maintenance_capex  free_cash_flow
0  2024-01-01 -2,784,026.56       0.00         478,605.94   -3,417,632.50
12 2025-01-01 -2,609,213.37 -96,674.86         485,777.00   -3,156,415.51
13 2025-02-01 -2,573,198.89   2,332.90         413,790.75   -3,147,422.55
24 2026-01-01 -2,540,602.36 -64,028.58         442,446.98   -3,080,220.76
1  2024-02-01 -2,329,202.06  33,660.63         400,551.16   -2,918,413.85
25 2026-02-01 -2,186,688.94  26,598.66         384,309.11   -2,758,796.71


## Conclusiones

Las conclusiones concretas se generan a partir de las salidas ejecutadas. Deben interpretarse como evidencia de una simulación y como demostración del método analítico.